> 在线讲义：[第 22 章 · 数值计算：积分器的阶、softmax 的上溢、半精度的累加和病态问题](https://wikieden.github.io/math-and-eval-stats/lessons/22-numerics.html) · 本书《数学与实验统计》

# 第 22 章 · 数值计算：积分器的阶、softmax 的上溢、半精度的累加和病态问题

配套讲义：仓库里 `math-and-eval-stats/notes/lessons/22-numerics.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy / SciPy（Colab 自带），不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一行代码、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 积分器 | Euler / Heun / RK4 | `integrate(m, n)` | 10 步：3.15e-2 / 4.01e-3 / 9.67e-7 |
| 第二步 softmax | 减最大值 | `softmax_stable(z)`、`logsumexp(z)` | [0.09, 0.2447, 0.6652]；1002.4076 |
| 第三步 累加 | 不同精度 | `accumulate(np.float16)`、`accumulate_bf16()` | 0.25、0.03125 |
| 第四步 相消 | 改写 | `cancel(1e-8)` | 0 vs 5e-17 |
| 第四步 病态 | 条件数 | `hilbert_cond(12)` | 1.6e16 |

例子：ODE x' = cos(t)·x 从 t = 0 积到 1（精确解 e^{sin 1}）；softmax 的上溢；float16 / bfloat16 / float32 的累加；1 − cos x 的相消；Hilbert 矩阵的病态。
里程碑 ME-M18 的"同一个 ODE 画步数 → 误差图"留给自己做。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import math
import numpy as np
EXACT = math.exp(math.sin(1.0))  # x' = cos(t)·x、x(0) = 1 在 t = 1 的精确值

**函数 `f`**：小工具，把结果整理成好读的样子再打印。

In [ ]:
def f(t: float, x: float) -> float:
    return math.cos(t) * x

**函数 `integrate`**：从 t = 0 到 1 走 n 步：Euler（1 阶）、Heun（2 阶）、RK4（4 阶）。

In [ ]:
def integrate(method: str, n: int) -> float:
    """从 t = 0 到 1 走 n 步：Euler（1 阶）、Heun（2 阶）、RK4（4 阶）。"""
    h, t, x = 1.0 / n, 0.0, 1.0
    for _ in range(n):
        if method == "euler":
            x = x + h * f(t, x)
        elif method == "heun":
            k1 = f(t, x)
            k2 = f(t + h, x + h * k1)
            x = x + h / 2 * (k1 + k2)
        else:
            k1 = f(t, x)
            k2 = f(t + h / 2, x + h / 2 * k1)
            k3 = f(t + h / 2, x + h / 2 * k2)
            k4 = f(t + h, x + h * k3)
            x = x + h / 6 * (k1 + 2 * k2 + 2 * k3 + k4)
        t += h
    return x

**函数 `softmax_naive`**：小工具，把结果整理成好读的样子再打印。

In [ ]:
def softmax_naive(z: np.ndarray) -> np.ndarray:
    e = np.exp(z)
    return e / e.sum()

**函数 `softmax_stable`**：先减最大值：e^{z − max} 不会上溢，结果不变。

In [ ]:
def softmax_stable(z: np.ndarray) -> np.ndarray:
    """先减最大值：e^{z − max} 不会上溢，结果不变。"""
    e = np.exp(z - z.max())
    return e / e.sum()

**函数 `logsumexp`**：log Σ e^z = max + log Σ e^{z − max}。

In [ ]:
def logsumexp(z: np.ndarray) -> float:
    """log Σ e^z = max + log Σ e^{z − max}。"""
    m = z.max()
    return float(m + np.log(np.exp(z - m).sum()))

**函数 `accumulate`**：在给定精度里把 step 加 n 次（真值 1.0）。

In [ ]:
def accumulate(dtype, n: int = 10000, step: float = 1e-4) -> float:
    """在给定精度里把 step 加 n 次（真值 1.0）。"""
    s = np.array(0.0, dtype=dtype)
    d = np.array(step, dtype=dtype)
    for _ in range(n):
        s = (s + d).astype(dtype)
    return float(s)

**函数 `bf16`**：用 float32 截掉低 16 位模拟 bfloat16（8 位指数、7 位尾数，就近舍入）。

In [ ]:
def bf16(x: np.ndarray) -> np.ndarray:
    """用 float32 截掉低 16 位模拟 bfloat16（8 位指数、7 位尾数，就近舍入）。"""
    b = np.asarray(x, np.float32).view(np.uint32)
    b = ((b + 0x7FFF + ((b >> 16) & 1)) & 0xFFFF0000).astype(np.uint32)
    return b.view(np.float32)

**函数 `accumulate_bf16`**：小工具，把结果整理成好读的样子再打印。

In [ ]:
def accumulate_bf16(n: int = 10000, step: float = 1e-4) -> float:
    s = np.float32(0.0)
    d = bf16(np.float32(step))
    for _ in range(n):
        s = bf16(np.float32(s + d))
    return float(s)

**函数 `cancel`**：1 − cos x 直接算 vs 换成 2 sin²(x/2)。

In [ ]:
def cancel(x: float) -> tuple[float, float]:
    """1 − cos x 直接算 vs 换成 2 sin²(x/2)。"""
    return 1 - math.cos(x), 2 * math.sin(x / 2) ** 2

**函数 `hilbert_cond`**：小工具，把结果整理成好读的样子再打印。

In [ ]:
def hilbert_cond(n: int) -> float:
    h = np.array([[1 / (i + j + 1) for j in range(n)] for i in range(n)])
    return float(np.linalg.cond(h))

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 ODE 积分器：阶数决定步数翻倍时误差降多少")
for n in (5, 10, 20):
    errs = [abs(integrate(m, n) - EXACT) for m in ("euler", "heun", "rk4")]
    print(f"  {n:2d} 步：Euler {errs[0]:.2e}，Heun {errs[1]:.2e}，RK4 {errs[2]:.2e}")
print("  步数翻倍：Euler 误差约 ÷2，Heun 约 ÷4，RK4 约 ÷16")

In [ ]:
print("\n第二步 softmax 和 logsumexp：先减最大值")
z = np.array([1000.0, 1001.0, 1002.0])
with np.errstate(over="ignore", invalid="ignore"):
    print(f"  直接算 softmax([1000, 1001, 1002]) = {softmax_naive(z).tolist()}")
print(f"  先减最大值 = {np.round(softmax_stable(z), 4).tolist()}；logsumexp = {logsumexp(z):.4f}")

In [ ]:
print("\n第三步 浮点精度：把 0.0001 加一万次")
print(f"  float32：{accumulate(np.float32):.6f}；float16：{accumulate(np.float16):.6f}；bfloat16：{accumulate_bf16():.6f}")
print(f"  float16 最大 {np.finfo(np.float16).max:.0f}、最小正规数 {np.finfo(np.float16).tiny:.1e}；bfloat16 的范围和 float32 一样（约 3.4e38），但只有 3 位有效数字")

In [ ]:
print("\n第四步 相减相消：两个差不多的数相减，有效数字丢光")
for x in (1e-3, 1e-8):
    a, b = cancel(x)
    print(f"  x = {x}：1 − cos x = {a:.6e}，2 sin²(x/2) = {b:.6e}")

In [ ]:
print("\n第五步 病态问题：Hilbert 矩阵")
for n in (4, 8, 12):
    print(f"  {n}×{n}：条件数 {hilbert_cond(n):.1e}")

## 3. 代码里才看得到的

1. **NumPy 没有 bfloat16**：代码把 float32 的位模式截掉低 16 位来模拟 bf16（bf16 就是 fp32 的高 16 位），每次加法后都重新截一次。
2. **`np.errstate` 屏蔽警告**：直接算 softmax 时 e^{1000} 上溢会报 RuntimeWarning，代码只是为了演示才故意这么写。
3. **RK4 每步调 4 次 f**：比较"20 步 RK4"和"20 步 Euler"不公平；按调用次数比，要拿 RK4 的 5 步和 Euler 的 20 步比（误差 1.55e-5 vs 1.56e-2）。

## 4. 自己改一改

1. 把累加的步长从 0.0001 改成 0.01，float16 能加到多少？
2. 给 `integrate` 加一个"中点法"（2 阶），看误差是不是也按 ÷4 下降。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 积分器有阶数，翻倍步数误差除二的 p 次方，采样步就是积分步；
> softmax 先减最大值，结果不变不上溢；
> 半精度加小数会被吞，累加和优化器留 fp32；
> 相近数别相减，病态问题算法救不了。